# eBATTLE Feature Engineering Pipeline

Builds a consolidated daily feature panel from raw Garmin wearable CSVs and trains binary classifiers to predict weekly physical activity target adherence.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, brier_score_loss
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print('✓ XGBoost available')
except ImportError:
    HAS_XGB = False
    print('✗ XGBoost not installed — skipping. pip install xgboost')

try:
    import shap
    HAS_SHAP = True
    print('✓ SHAP available')
except ImportError:
    HAS_SHAP = False
    print('✗ SHAP not installed — skipping. pip install shap')

print('All imports done.')

In [ ]:
NOTEBOOK_DIR = Path().resolve()
DATA_ROOT    = NOTEBOOK_DIR / 'Data'

CONFIG = {
    'pilot_dirs': {
        1: DATA_ROOT / 'pilot_utrekk1_271125',
        2: DATA_ROOT / 'pilot_utrekk2_271125',
    },
    'output_dir': NOTEBOOK_DIR / 'results_v3',

    'phase1_week_range': (0, 7),
    'phase2_week_range': (8, 60),

    'day1_source': 'first_signal',

    'wear_threshold_minutes': 600,   # min daily wear to count as valid (10 h)
    'min_wear_fraction':      0.60,  # Wearing Time (%) threshold
    'min_valid_days_per_user': 7,    # exclude users with fewer valid days
    'min_wear_days_per_week':  3,    # min valid days to score a week

    'rolling_recent_days':  7,  # window for recent_* features
    'min_obs_for_std':      3,  # minimum obs for cumul_std_*
    'min_obs_for_trend':    5,  # minimum obs for trend_*
    'min_history_days':     7,  # minimum history before first prediction row

    'max_missing_frac': 0.30,   # drop features with >30% missing
    'max_corr':         0.95,   # drop one of any pair with |r| > 0.95

    'cv_folds':    5,
    'cv_min_rows': 20,
    'random_state': 42,

    'user_col': 'user_id',
    'date_col': 'date',

    'targets': {
        'mvpa_75':  {'col': 'mvpa_min', 'agg': 'sum',  'threshold': 75,   'label': 'MVPA ≥75 min/wk'},
        'mvpa_150': {'col': 'mvpa_min', 'agg': 'sum',  'threshold': 150,  'label': 'MVPA ≥150 min/wk (WHO)'},
        'steps_5k': {'col': 'steps',    'agg': 'mean', 'threshold': 5000, 'label': 'Steps ≥5k/day avg'},
        'steps_7k': {'col': 'steps',    'agg': 'mean', 'threshold': 7000, 'label': 'Steps ≥7k/day avg'},
    },
}

CONFIG['phase1_days'] = (
    CONFIG['phase1_week_range'][0] * 7,
    CONFIG['phase1_week_range'][1] * 7 + 6,
)
CONFIG['phase2_days'] = (
    CONFIG['phase2_week_range'][0] * 7,
    CONFIG['phase2_week_range'][1] * 7 + 6,
)

CONFIG['output_dir'].mkdir(exist_ok=True)

print(f"Phase 1 : weeks {CONFIG['phase1_week_range']}  → days {CONFIG['phase1_days']}")
print(f"Phase 2 : weeks {CONFIG['phase2_week_range']} → days {CONFIG['phase2_days']}")
print(f"Output  : {CONFIG['output_dir'].resolve()}")

## 1. Data Ingestion

Loads `daily-health-log.csv` from both pilot directories, standardises column names, applies wear-time quality filters, and merges supplementary sleep efficiency and body-composition data into a single daily panel.

In [ ]:
CONFIG['min_valid_days_per_user'] = 3   # changed from 7 → 3

UNIT_MAP = {
    'steps':             'count',
    'mvpa_min':          'minutes',
    'kcal':              'kcal',
    'distance_m':        'metres',
    'hr_mean':           'bpm',
    'rhr_mean':          'bpm',
    'stress_mean':       '0–100 scale',
    'body_battery_avg':  '0–100 scale',
    'sleep_light_s':     'seconds',
    'sleep_deep_s':      'seconds',
    'sleep_rem_s':       'seconds',
    'sleep_awake_s':     'seconds',
    'rmssd_ms':          'milliseconds',
    'sdnn_ms':           'milliseconds',
    'spo2_mean':         'percent (0–100)',
    'resp_mean':         'breaths/min',
    'weight_kg':         'kilograms',
    'bmi':               'kg/m²',
    'body_fat_pct':      'percent',
    'wear_pct':          'percent (0–100)',
    'sleep_efficiency':  'percent (0–100)',
}
min_valid = CONFIG['min_valid_days_per_user']
print('CONFIG overrides applied:')
print(f'  min_valid_days_per_user = {min_valid}')
print(f'  Unit map defined for {len(UNIT_MAP)} features')

In [ ]:
DAILY_HEALTH_MAP = {
    'User Id':                   'user_id',
    'Calendar Date (Local)':     'date',
    'Wearing Time (s)':          'wear_seconds',
    'Wearing Time (%)':          'wear_pct',
    'Calories (kcal)':           'kcal',
    'Steps':                     'steps',
    'MVPA (mins)':               'mvpa_min',
    'Distance (m)':              'distance_m',
    'Floors Climbed':            'floors',
    'Mean Motion Intensity':     'motion_intensity',
    'MET (avg)':                 'met_avg',
    'Active Sec (s)':            'active_seconds',
    'Heart Rate (min)':          'hr_min',
    'Heart Rate (avg)':          'hr_mean',
    'Heart Rate (max)':          'hr_max',
    'Resting Heart Rate (avg)':  'rhr_mean',
    'Stress (avg)':              'stress_mean',
    'Stress (max)':              'stress_max',
    'Stress Qualifier':          'stress_qualifier',
    'Rest Stress (s)':           'stress_rest_s',
    'Low Stress (s)':            'stress_low_s',
    'Med Stress (s)':            'stress_med_s',
    'High Stress (s)':           'stress_high_s',
    'Body Battery (avg)':        'body_battery_avg',
    'Body Battery (min)':        'body_battery_min',
    'Body Battery (max)':        'body_battery_max',
    'Light Sleep Duration (s)':  'sleep_light_s',
    'Deep Sleep Duration (s)':   'sleep_deep_s',
    'Rem Sleep Duration (s)':    'sleep_rem_s',
    'Awake Duration (s)':        'sleep_awake_s',
    'Dominant Sleep Phase':      'sleep_dominant_phase',
    'RMSSD':                     'rmssd_ms',
    'SDNN':                      'sdnn_ms',
    'BBI (avg)':                 'bbi_avg',
    'Pulse Ox (avg)':            'spo2_mean',
    'Respiration-Rate (avg)':    'resp_mean',
    'Skin Temperature (avg)':    'skin_temp_avg',
}

KEEP_COLS = list(DAILY_HEALTH_MAP.values())

# (exclude activity targets to avoid leakage in cumul_* features)
PHYSIOL_COLS = [
    'hr_mean', 'hr_min', 'hr_max',
    'rhr_mean',
    'stress_mean', 'stress_max',
    'stress_rest_s', 'stress_low_s', 'stress_med_s', 'stress_high_s',
    'body_battery_avg', 'body_battery_min', 'body_battery_max',
    'sleep_light_s', 'sleep_deep_s', 'sleep_rem_s', 'sleep_awake_s',
    'rmssd_ms', 'sdnn_ms', 'bbi_avg',
    'spo2_mean',
    'resp_mean',
    'motion_intensity', 'met_avg', 'active_seconds',
    'wear_pct',
    'sleep_efficiency',   # from sleep.csv
    'weight_kg',          # from body-composition.csv
    'bmi',                # from body-composition.csv
    'body_fat_pct',       # from body-composition.csv
]

ACTIVITY_FEAT_COLS = ['steps', 'mvpa_min', 'kcal', 'distance_m', 'floors']

ALL_FEAT_COLS = PHYSIOL_COLS + ACTIVITY_FEAT_COLS

print(f'Physiol features: {len(PHYSIOL_COLS)}')
print(f'Activity features: {len(ACTIVITY_FEAT_COLS)}')
print(f'Total feature base: {len(ALL_FEAT_COLS)}')

In [ ]:
def load_daily_health(pilot_dirs: dict) -> pd.DataFrame:
    """Load daily-health-log.csv from all pilots, standardise columns,
    apply data quality filters, and return a merged DataFrame."""
    frames = []
    quality_log = []

    for pilot_id, pilot_dir in pilot_dirs.items():
        fpath = pilot_dir / 'daily-health-log.csv'
        if not fpath.exists():
            print(f'  [P{pilot_id}] daily-health-log.csv NOT FOUND — skipping')
            continue

        raw = pd.read_csv(fpath, low_memory=False)
        n_raw = len(raw)

        raw = raw.rename(columns=DAILY_HEALTH_MAP)
        raw = raw[[c for c in KEEP_COLS if c in raw.columns]].copy()
        raw['pilot_id'] = pilot_id

        raw['date'] = pd.to_datetime(raw['date'], errors='coerce').dt.date
        raw = raw.dropna(subset=['user_id', 'date'])

        for col in ['hr_mean', 'hr_min', 'hr_max']:
            if col in raw.columns:
                raw.loc[raw[col] == 0, col] = np.nan

        no_wear = raw.get('wear_pct', pd.Series(dtype=float)) == 0
        for col in ['steps', 'mvpa_min']:
            if col in raw.columns:
                raw.loc[no_wear, col] = np.nan

        if 'wear_pct' in raw.columns:
            raw['valid_wear_day'] = (
                raw['wear_pct'].fillna(0) >= CONFIG['min_wear_fraction'] * 100
            ).astype(int)
        else:
            raw['valid_wear_day'] = 0

        n_valid = raw['valid_wear_day'].sum()
        n_dropped = n_raw - len(raw)

        quality_log.append({
            'pilot': pilot_id,
            'rows_raw': n_raw,
            'rows_kept': len(raw),
            'rows_dropped_bad_date': n_dropped,
            'valid_wear_days': n_valid,
            'users': raw['user_id'].nunique(),
        })

        frames.append(raw)
        print(f'  [P{pilot_id}] {len(raw):,} rows | {raw["user_id"].nunique()} users | {n_valid:,} valid-wear days')

    if not frames:
        raise RuntimeError('No data loaded — check pilot_dirs in CONFIG')

    df = pd.concat(frames, ignore_index=True)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

    df._quality_log = pd.DataFrame(quality_log)
    return df

print('Loading daily-health-log.csv ...')
df_daily = load_daily_health(CONFIG['pilot_dirs'])
print(f'\nTotal: {len(df_daily):,} rows | {df_daily["user_id"].nunique()} users')

In [ ]:
SIGNAL_FILES = [
    'heart-rate.csv',
    'stress.csv',
    'respiration.csv',
    'sleep.csv',
    'rmssd.csv',
    'dailies.csv',
    'daily-health-log.csv',
]

DATE_COLS_CANDIDATES = [
    'Calendar Date (Local)', 'Calendar Date (UTC)',
    'Start Time (Local)', 'Start Time (UTC)',
]

def compute_day1_per_user(pilot_dirs: dict) -> pd.Series:
    """Return a Series: user_id → first signal date (as Timestamp)."""
    records = {}  # user_id → earliest date seen

    for pilot_id, pilot_dir in pilot_dirs.items():
        for fname in SIGNAL_FILES:
            fpath = pilot_dir / fname
            if not fpath.exists():
                continue
            try:
                chunk = pd.read_csv(fpath, usecols=lambda c: c in ['User Id'] + DATE_COLS_CANDIDATES,
                                    low_memory=False, nrows=None)
            except Exception:
                continue

            if 'User Id' not in chunk.columns:
                continue

            date_col = next((c for c in DATE_COLS_CANDIDATES if c in chunk.columns), None)
            if date_col is None:
                continue

            chunk['_date'] = pd.to_datetime(chunk[date_col], errors='coerce').dt.normalize()
            chunk = chunk.dropna(subset=['User Id', '_date'])

            for uid, grp in chunk.groupby('User Id'):
                earliest = grp['_date'].min()
                if uid not in records or earliest < records[uid]:
                    records[uid] = earliest

    day1 = pd.Series(records, name='day1_date')
    day1.index.name = 'user_id'
    print(f'Day 1 computed for {len(day1)} users')
    return day1

print('Computing Day 1 per user from signal files ...')
day1_map = compute_day1_per_user(CONFIG['pilot_dirs'])
print(day1_map.sort_values().head(10).to_string())

In [ ]:
def add_phase_labels(df: pd.DataFrame, day1_map: pd.Series) -> pd.DataFrame:
    """Add days_since_day1, phase_num, phase_label columns."""
    df = df.copy()

    df['day1_date'] = df['user_id'].map(day1_map)
    df['days_since_day1'] = (
        df['date'] - df['day1_date']
    ).dt.days

    p1_lo, p1_hi = CONFIG['phase1_days']
    p2_lo, p2_hi = CONFIG['phase2_days']

    df['phase_label'] = 'pre_trial'
    df['phase_num']   = 0

    mask_p1 = df['days_since_day1'].between(p1_lo, p1_hi)
    mask_p2 = df['days_since_day1'].between(p2_lo, p2_hi)
    mask_post = df['days_since_day1'] > p2_hi
    mask_pre  = df['days_since_day1'] < p1_lo

    df.loc[mask_p1,   'phase_label'] = 'phase1'
    df.loc[mask_p1,   'phase_num']   = 1
    df.loc[mask_p2,   'phase_label'] = 'phase2'
    df.loc[mask_p2,   'phase_num']   = 2
    df.loc[mask_post, 'phase_label'] = 'post_trial'
    df.loc[mask_post, 'phase_num']   = 3
    df.loc[mask_pre,  'phase_label'] = 'pre_trial'
    df.loc[mask_pre,  'phase_num']   = 0

    df.loc[df['day1_date'].isna(), ['phase_label', 'phase_num']] = ['unknown', np.nan]

    print('Phase distribution:')
    print(df['phase_label'].value_counts().to_string())
    return df

df_daily = add_phase_labels(df_daily, day1_map)

In [ ]:
def load_sleep_efficiency(pilot_dirs: dict) -> pd.DataFrame:
    """Daily sleep efficiency and sleep score per user from sleep.csv."""
    frames = []
    for pilot_id, pilot_dir in pilot_dirs.items():
        fpath = pilot_dir / 'sleep.csv'
        if not fpath.exists():
            continue
        s = pd.read_csv(fpath, low_memory=False)
        if 'User Id' not in s.columns:
            continue
        date_col = next((c for c in ['Calendar Date (Local)', 'Calendar Date (UTC)'] if c in s.columns), None)
        if date_col is None:
            continue
        s['date'] = pd.to_datetime(s[date_col], errors='coerce').dt.normalize()
        s = s.rename(columns={
            'User Id': 'user_id',
            'Sleep Efficiency': 'sleep_efficiency',
            'Sleep Score Value': 'sleep_score',
        })
        keep = ['user_id', 'date', 'sleep_efficiency', 'sleep_score']
        s = s[[c for c in keep if c in s.columns]].dropna(subset=['user_id', 'date'])
        s = s.groupby(['user_id', 'date'], as_index=False).agg(
            sleep_efficiency=('sleep_efficiency', 'mean'),
            sleep_score=('sleep_score', 'mean') if 'sleep_score' in s.columns else ('date', 'count'),
        )
        frames.append(s)
    if not frames:
        return pd.DataFrame(columns=['user_id', 'date', 'sleep_efficiency'])
    return pd.concat(frames, ignore_index=True)

def load_body_composition(pilot_dirs: dict) -> pd.DataFrame:
    """Most recent body composition per user per date (forward-filled for merging)."""
    frames = []
    for pilot_id, pilot_dir in pilot_dirs.items():
        fpath = pilot_dir / 'body-composition.csv'
        if not fpath.exists():
            continue
        b = pd.read_csv(fpath, low_memory=False)
        if 'User Id' not in b.columns or 'Weight (g)' not in b.columns:
            continue
        date_col = next((c for c in ['Calendar Date (Local)', 'Calendar Date (UTC)'] if c in b.columns), None)
        if date_col is None:
            continue
        b['date'] = pd.to_datetime(b[date_col], errors='coerce').dt.normalize()
        b = b.rename(columns={
            'User Id':          'user_id',
            'Weight (g)':       'weight_g',
            'Body Fat (%)':     'body_fat_pct',
            'Body Mass Index':  'bmi',
            'Muscle Mass (g)':  'muscle_mass_g',
            'Body Water (%)':   'body_water_pct',
        })
        b['weight_kg'] = b['weight_g'] / 1000
        keep = ['user_id', 'date', 'weight_kg', 'bmi', 'body_fat_pct', 'muscle_mass_g', 'body_water_pct']
        b = b[[c for c in keep if c in b.columns]].dropna(subset=['user_id', 'date', 'weight_kg'])
        b = b[b['weight_kg'] > 0]
        frames.append(b)
    if not frames:
        return pd.DataFrame(columns=['user_id', 'date', 'weight_kg', 'bmi', 'body_fat_pct'])
    return pd.concat(frames, ignore_index=True)

print('Loading supplementary files ...')
df_sleep = load_sleep_efficiency(CONFIG['pilot_dirs'])
df_bodycomp = load_body_composition(CONFIG['pilot_dirs'])
print(f'  sleep.csv    : {len(df_sleep):,} user-days with sleep efficiency')
print(f'  body-comp.csv: {len(df_bodycomp):,} weighings from {df_bodycomp["user_id"].nunique()} users')

In [ ]:
def validate_panel(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Deduplicate on (user_id, date) composite key — keep row with most non-null values.
    2. Validate numeric ranges and replace implausible values with NaN.
    3. Ensure date column is timezone-naive (local date).
    """
    df = df.copy()
    n_before = len(df)

    df['_nonull'] = df.select_dtypes(include='number').notna().sum(axis=1)
    df = (df.sort_values('_nonull', ascending=False)
            .drop_duplicates(subset=['user_id', 'date'], keep='first')
            .drop(columns='_nonull'))
    n_dedup = n_before - len(df)
    if n_dedup > 0:
        print(f'  Deduplication: removed {n_dedup} duplicate (user_id, date) rows')

    checks = {
        'hr_mean':          (30, 220),   # bpm
        'stress_mean':      (0,  100),   # Garmin 0–100 scale
        'body_battery_avg': (0,  100),   # Garmin 0–100 scale
        'spo2_mean':        (70, 100),   # SpO2 %
        'resp_mean':        (4,  60),    # breaths/min
        'rmssd_ms':         (0,  300),   # ms
        'wear_pct':         (0,  100),   # percent
        'weight_kg':        (20, 300),   # kg (paediatric → extreme obesity)
        'bmi':              (10, 80),    # kg/m²
        'sleep_efficiency': (0,  100),   # percent
    }
    flags = {}
    for col, (lo, hi) in checks.items():
        if col not in df.columns:
            continue
        out = ((df[col] < lo) | (df[col] > hi)) & df[col].notna()
        n_out = int(out.sum())
        if n_out > 0:
            flags[col] = n_out
            df.loc[out, col] = np.nan

    if flags:
        print('  Implausible values → NaN:')
        for col, n in flags.items():
            print(f'    {col}: {n} rows')
    else:
        print('  All plausibility checks passed.')

    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)

    print(f'  Panel after validation: {len(df):,} rows | {df["user_id"].nunique()} users')
    return df

print('validate_panel() defined — will be called after merge_panel.')

In [ ]:
def merge_panel(df_daily, df_sleep, df_bodycomp):
    """Merge supplementary files onto the daily panel.
    Body composition is forward-filled (most recent measurement applies).
    Calls validate_panel() at the end for dedup + plausibility checks."""
    df = df_daily.copy()

    if len(df_sleep) > 0:
        df_sleep = df_sleep.copy()
        df_sleep['date'] = pd.to_datetime(df_sleep['date'])
        df = df.merge(df_sleep, on=['user_id', 'date'], how='left')
    else:
        df['sleep_efficiency'] = np.nan

    if len(df_bodycomp) > 0:
        df_bodycomp = df_bodycomp.copy()
        df_bodycomp['date'] = pd.to_datetime(df_bodycomp['date'])
        bc_cols = [c for c in ['weight_kg', 'bmi', 'body_fat_pct', 'muscle_mass_g', 'body_water_pct']
                   if c in df_bodycomp.columns]
        df = df.merge(df_bodycomp[['user_id', 'date'] + bc_cols], on=['user_id', 'date'], how='left')
        df = df.sort_values(['user_id', 'date'])
        for col in bc_cols:
            df[col] = df.groupby('user_id')[col].transform(lambda s: s.ffill())
    else:
        for col in ['weight_kg', 'bmi', 'body_fat_pct']:
            df[col] = np.nan

    df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

    valid_counts = df.groupby('user_id')['valid_wear_day'].sum()
    valid_users = valid_counts[valid_counts >= CONFIG['min_valid_days_per_user']].index
    n_excluded = df['user_id'].nunique() - len(valid_users)
    df = df[df['user_id'].isin(valid_users)].reset_index(drop=True)

    print(f'Panel: {len(df):,} rows | {df["user_id"].nunique()} users ({n_excluded} excluded <{CONFIG["min_valid_days_per_user"]} valid days)')
    print(f'Pilots: {df.groupby("pilot_id")["user_id"].nunique().to_dict()}')
    print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')

    print('Running validation ...')
    df = validate_panel(df)
    return df

df_panel = merge_panel(df_daily, df_sleep, df_bodycomp)
df_panel.head(3)

In [ ]:
feat_coverage = (
    df_panel[[c for c in ALL_FEAT_COLS if c in df_panel.columns]]
    .notna().mean().sort_values(ascending=False)
)

print('Feature coverage (fraction non-null):')
print(feat_coverage.round(3).to_string())

quality_df = pd.DataFrame({
    'feature': feat_coverage.index,
    'coverage': feat_coverage.values,
    'missing_pct': (1 - feat_coverage.values) * 100,
})
quality_df.to_csv(CONFIG['output_dir'] / 'data_quality_report.csv', index=False)
print(f"\nSaved: {CONFIG['output_dir'] / 'data_quality_report.csv'}")

## 2. Cumulative Feature Engineering

For each user–week, five feature types are derived per signal using all history up to the end of that week:

| Prefix | Description |
|---|---|
| `recent_` | Mean over the preceding 7 days |
| `cumul_mean_` | Mean over all available history |
| `cumul_std_` | Standard deviation over all available history |
| `trend_` | Linear slope over all available history (via `numpy.polyfit`) |
| `delta_` | Difference between `recent_` and `cumul_mean_` |

In [ ]:
def compute_trend(values: np.ndarray) -> float:
    """Linear slope (per day) via numpy.polyfit. Returns NaN if too few obs."""
    valid = values[~np.isnan(values)]
    if len(valid) < CONFIG['min_obs_for_trend']:
        return np.nan
    x = np.arange(len(valid), dtype=float)
    try:
        slope, _ = np.polyfit(x, valid, 1)
        return slope
    except Exception:
        return np.nan

def compute_cumulative_features(df: pd.DataFrame, feat_cols: list) -> pd.DataFrame:
    """
    Build one row per (user_id, year_week) with cumulative features.
    Uses all history up to and including the end of that week.
    """
    feat_cols = [c for c in feat_cols if c in df.columns]

    df = df.copy()
    df['year_week'] = df['date'].dt.isocalendar().year.astype(str) + '_' + \
                      df['date'].dt.isocalendar().week.astype(str).str.zfill(2)

    rows = []
    recent_days = CONFIG['rolling_recent_days']

    for user_id, user_df in df.groupby('user_id'):
        user_df = user_df.sort_values('date').reset_index(drop=True)
        weeks = user_df['year_week'].unique()

        for week in weeks:
            week_mask = user_df['year_week'] == week
            week_end_idx = week_mask[week_mask].index.max()

            hist = user_df.loc[:week_end_idx]
            recent = hist.tail(recent_days)

            n_history = len(hist)
            if n_history < CONFIG['min_history_days']:
                continue  # Skip — not enough history for meaningful features

            row = {
                'user_id':         user_id,
                'year_week':       week,
                'days_since_day1': hist['days_since_day1'].max(),
                'n_data_days':     n_history,
                'n_recent_days':   len(recent),
                'phase_num':       hist['phase_num'].iloc[-1],
                'phase_label':     hist['phase_label'].iloc[-1],
                'pilot_id':        hist['pilot_id'].iloc[-1],
                'valid_wear_days': hist['valid_wear_day'].sum(),
            }

            for col in feat_cols:
                h_vals = hist[col].values.astype(float)
                r_vals = recent[col].values.astype(float)

                h_valid = h_vals[~np.isnan(h_vals)]
                r_valid = r_vals[~np.isnan(r_vals)]

                c_mean = np.nanmean(h_vals) if len(h_valid) > 0 else np.nan
                r_mean = np.nanmean(r_vals) if len(r_valid) > 0 else np.nan

                row[f'recent_{col}']     = r_mean
                row[f'cumul_mean_{col}'] = c_mean
                row[f'cumul_std_{col}']  = np.nanstd(h_vals, ddof=1) if len(h_valid) >= CONFIG['min_obs_for_std'] else np.nan
                row[f'trend_{col}']      = compute_trend(h_vals)
                row[f'delta_{col}']      = (r_mean - c_mean) if (not np.isnan(r_mean) and not np.isnan(c_mean)) else np.nan

            rows.append(row)

    cumul_df = pd.DataFrame(rows)
    print(f'Cumulative features: {len(cumul_df):,} rows (user-weeks) | {cumul_df.shape[1]} columns')
    return cumul_df

print('Building cumulative features (this may take a minute)...')
cumul_df = compute_cumulative_features(df_panel, ALL_FEAT_COLS)
cumul_df.head(3)

In [ ]:
LOAD_FROM_CACHE = False  # ← set True after first run to reload instantly

CACHE_PATH = CONFIG['output_dir'] / 'cumul_features_cache.parquet'

if not LOAD_FROM_CACHE:
    cumul_df.to_parquet(CACHE_PATH, index=False)
    print(f'Saved cumulative feature cache → {CACHE_PATH}')
    print(f'  Shape: {cumul_df.shape} | Users: {cumul_df["user_id"].nunique()}')
else:
    if CACHE_PATH.exists():
        cumul_df = pd.read_parquet(CACHE_PATH)
        print(f'Loaded from cache → {CACHE_PATH}')
        print(f'  Shape: {cumul_df.shape} | Users: {cumul_df["user_id"].nunique()}')
    else:
        print(f'Cache not found at {CACHE_PATH} — run with LOAD_FROM_CACHE=False first')

## 3. Weekly Activity Targets

Aggregates daily records to weekly totals and computes binary targets (met / not met) for four activity thresholds. Each target is shifted forward by one week so that features at week *t* predict adherence at week *t*+1.

In [ ]:
def build_weekly_targets(df_panel: pd.DataFrame) -> pd.DataFrame:
    """Aggregate activity targets per user-week and add next-week labels."""
    df = df_panel.copy()
    df['year_week'] = df['date'].dt.isocalendar().year.astype(str) + '_' + \
                      df['date'].dt.isocalendar().week.astype(str).str.zfill(2)

    target_frames = []

    for user_id, user_df in df.groupby('user_id'):
        user_df = user_df.sort_values('date')

        weekly = user_df.groupby('year_week').agg(
            mvpa_sum   =('mvpa_min',       'sum'),
            steps_mean =('steps',          'mean'),
            valid_days =('valid_wear_day', 'sum'),
        ).reset_index()

        weekly = weekly[weekly['valid_days'] >= CONFIG['min_wear_days_per_week']]

        for tgt_name, tgt_cfg in CONFIG['targets'].items():
            col   = 'mvpa_sum' if tgt_cfg['col'] == 'mvpa_min' else 'steps_mean'
            thr   = tgt_cfg['threshold']
            weekly[f'target_{tgt_name}'] = (weekly[col] >= thr).astype(int)
            # Shift forward: next week's target
            weekly[f'next_{tgt_name}'] = weekly[f'target_{tgt_name}'].shift(-1)

        weekly['user_id'] = user_id
        weekly = weekly.iloc[:-1]
        target_frames.append(weekly)

    target_df = pd.concat(target_frames, ignore_index=True)

    print('Target class balance:')
    for tgt_name in CONFIG['targets']:
        col = f'next_{tgt_name}'
        if col in target_df.columns:
            vc = target_df[col].value_counts(normalize=True)
            print(f"  {tgt_name:10s}: active={vc.get(1.0,0):.1%}  inactive={vc.get(0.0,0):.1%}  "
                  f"n={target_df[col].notna().sum()}")

    return target_df

target_df = build_weekly_targets(df_panel)

target_cols = ['user_id', 'year_week'] + [c for c in target_df.columns if c.startswith('next_')]
model_df = cumul_df.merge(target_df[target_cols], on=['user_id', 'year_week'], how='inner')
print(f'\nModel dataset: {len(model_df):,} user-weeks | {model_df["user_id"].nunique()} users')

## 4. Feature Selection

Removes features with more than 30% missing values, then removes one feature from each pair with Pearson |r| > 0.95.

In [ ]:
def select_features(df: pd.DataFrame) -> list:
    """Return feature column list after missingness and correlation filtering."""
    prefixes = ('recent_', 'cumul_mean_', 'cumul_std_', 'trend_', 'delta_')
    meta_feats = ['days_since_day1', 'n_data_days', 'n_recent_days', 'valid_wear_days', 'phase_num']

    all_feats = [c for c in df.columns if c.startswith(prefixes)] + \
                [c for c in meta_feats if c in df.columns]

    print(f'Candidate features: {len(all_feats)}')

    missing = df[all_feats].isna().mean()
    keep = missing[missing <= CONFIG['max_missing_frac']].index.tolist()
    dropped_missing = len(all_feats) - len(keep)
    print(f'After missingness filter (>{CONFIG["max_missing_frac"]:.0%}): {len(keep)} features ({dropped_missing} dropped)')

    if len(keep) > 1:
        corr = df[keep].corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > CONFIG['max_corr'])]
        keep = [c for c in keep if c not in to_drop]
        print(f'After correlation filter (|r|>{CONFIG["max_corr"]}): {len(keep)} features ({len(to_drop)} dropped)')

    for pfx in prefixes:
        n = sum(1 for c in keep if c.startswith(pfx))
        print(f'  {pfx:<15s}: {n}')
    n_meta = sum(1 for c in keep if c in meta_feats)
    print(f'  {"meta":<15s}: {n_meta}')

    return keep

feature_list = select_features(model_df)
print(f'\nFinal feature count: {len(feature_list)}')

In [ ]:
for pid, label in [(1, 'Pilot 1'), (2, 'Pilot 2')]:
    users_pid = model_df[model_df['pilot_id'] == pid]['user_id'].value_counts()
    if users_pid.empty:
        print(f'No {label} users in model dataset')
        continue
    uid = users_pid.index[0]
    user_rows = model_df[model_df['user_id'] == uid]
    print(f'\n{label} — User {uid[:12]}... ({len(user_rows)} weeks)')
    sample_cols = ['days_since_day1', 'n_data_days', 'phase_label']
    sample_cols += [c for c in feature_list if 'hr_mean' in c or 'stress_mean' in c][:6]
    print(user_rows[sample_cols].head(4).to_string(index=False))

## 5. Evaluation Metrics

| Metric | Range | Chance level |
|---|---|---|
| ROC AUC | 0–1 | 0.50 |
| Balanced Accuracy | 0–1 | 0.50 |
| F1 Macro | 0–1 | ~0.50 |
| F1 Active | 0–1 | varies |
| Brier Score | 0–0.25 | 0.25 (lower is better) |

## 6. Model Definitions

Defines four classifiers: Logistic Regression, Random Forest, Gradient Boosting, and XGBoost. Random Forest and XGBoost are wrapped in `RandomizedSearchCV` for hyperparameter tuning.

In [ ]:
RS = CONFIG['random_state']

MODELS = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     LogisticRegression(
            C=1.0, class_weight='balanced',
            solver='lbfgs', max_iter=2000, random_state=RS
        )),
    ]),

    'RandomForest': RandomizedSearchCV(
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf',     RandomForestClassifier(
                class_weight='balanced', random_state=RS, n_jobs=-1
            )),
        ]),
        param_distributions={
            'clf__n_estimators':   [100, 200, 300],
            'clf__max_depth':      [3, 4, 6, None],
            'clf__min_samples_leaf': [4, 8, 16],
            'clf__max_features':   ['sqrt', 0.5, 0.7],
        },
        n_iter=10, cv=3, scoring='roc_auc',
        random_state=RS, n_jobs=-1, refit=True
    ),

    'GradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf',     GradientBoostingClassifier(
            n_estimators=100, max_depth=3,
            learning_rate=0.05, subsample=0.8,
            random_state=RS
        )),
    ]),
}

if HAS_XGB:
    MODELS['XGBoost'] = RandomizedSearchCV(
        XGBClassifier(
            use_label_encoder=False,
            eval_metric='logloss',
            scale_pos_weight=1,
            random_state=RS,
            n_jobs=-1
        ),
        param_distributions={
            'n_estimators':   [100, 200, 300],
            'max_depth':      [3, 4, 6],
            'learning_rate':  [0.01, 0.05, 0.1],
            'subsample':      [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
        },
        n_iter=10, cv=3, scoring='roc_auc',
        random_state=RS, n_jobs=-1, refit=True
    )

print(f'Models defined: {", ".join(MODELS.keys())}')

## 7. Cross-Validated Evaluation

Evaluates all models using 5-fold `StratifiedGroupKFold` (grouped by `user_id`) across all four activity targets.

In [ ]:
def evaluate_target(data, feature_list, target_col, description, models):
    """5-fold StratifiedGroupKFold CV for all models on one target.
    Returns results DataFrame and per-fold feature importances for RF."""

    sub = data.dropna(subset=[target_col]).copy()
    avail_feats = [f for f in feature_list if f in sub.columns]

    if len(sub) < CONFIG['cv_min_rows']:
        print(f'  Skipping {description}: only {len(sub)} rows')
        return pd.DataFrame(), {}
    if sub[target_col].nunique() < 2:
        print(f'  Skipping {description}: single class')
        return pd.DataFrame(), {}

    X = sub[avail_feats].values
    y = sub[target_col].astype(int).values
    groups = sub['user_id'].values

    cv = StratifiedGroupKFold(n_splits=CONFIG['cv_folds'], shuffle=True,
                              random_state=CONFIG['random_state'])

    results = []
    fold_importances = {m: [] for m in models}  # RF / XGB fold-level importances

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        for model_name, model in models.items():
            try:
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_te)
                y_prob = model.predict_proba(X_te)[:, 1] \
                         if hasattr(model, 'predict_proba') \
                         else model.decision_function(X_te)

                result = {
                    'target':      description,
                    'model':       model_name,
                    'fold':        fold,
                    'n_test':      len(y_te),
                    'bal_acc':     balanced_accuracy_score(y_te, y_pred),
                    'f1_macro':    f1_score(y_te, y_pred, average='macro', zero_division=0),
                    'f1_active':   f1_score(y_te, y_pred, pos_label=1, zero_division=0),
                    'precision':   precision_score(y_te, y_pred, pos_label=1, zero_division=0),
                    'recall':      recall_score(y_te, y_pred, pos_label=1, zero_division=0),
                    'roc_auc':     roc_auc_score(y_te, y_prob) if len(np.unique(y_te)) > 1 else np.nan,
                    'brier_score': brier_score_loss(y_te, np.clip(y_prob, 0, 1)),
                }
                results.append(result)

                clf = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
                if hasattr(clf, 'named_steps'):
                    clf = clf.named_steps.get('clf', clf)
                if hasattr(clf, 'feature_importances_'):
                    fold_importances[model_name].append(clf.feature_importances_)

            except Exception as e:
                print(f'    {model_name} fold {fold} failed: {e}')

    results_df = pd.DataFrame(results)
    return results_df, fold_importances

print('Evaluation function ready.')

In [ ]:
all_results   = []
all_fold_imps = {}  # target → model → list of fold importance arrays
best_target   = None
best_auc      = -1

for tgt_name, tgt_cfg in CONFIG['targets'].items():
    col = f'next_{tgt_name}'
    if col not in model_df.columns:
        print(f'Target {col} not found — skipping')
        continue

    print(f'\n── {tgt_cfg["label"]} ──')
    res_df, fold_imps = evaluate_target(
        model_df, feature_list, col, tgt_cfg['label'], MODELS
    )

    if len(res_df) == 0:
        continue

    all_results.append(res_df)
    all_fold_imps[tgt_name] = fold_imps

    mean_auc = res_df.groupby('model')['roc_auc'].mean().max()
    if mean_auc > best_auc:
        best_auc    = mean_auc
        best_target = tgt_name

combined = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
print(f'\nBest target: {best_target} (mean ROC AUC = {best_auc:.3f})')

In [ ]:
if len(combined) > 0:
    summary = combined.groupby(['target', 'model']).agg(
        bal_acc   =('bal_acc',    'mean'),
        f1_macro  =('f1_macro',   'mean'),
        f1_active =('f1_active',  'mean'),
        roc_auc   =('roc_auc',    'mean'),
        brier     =('brier_score','mean'),
    ).round(3)

    print('\n=== Results Summary (mean across folds) ===\n')
    print(summary.to_string())

    best_per_target = (
        summary.reset_index()
        .sort_values('roc_auc', ascending=False)
        .groupby('target').first()
        [['model', 'roc_auc', 'bal_acc', 'brier']]
    )
    print('\n=== Best model per target ===\n')
    print(best_per_target.to_string())

    summary.to_csv(CONFIG['output_dir'] / 'cumulative_results.csv')
    print(f"\nSaved: {CONFIG['output_dir'] / 'cumulative_results.csv'}")

## 8. Feature Importance

Computes feature importances by averaging across CV folds and aggregating across models using Borda count ranking.

In [ ]:
def borda_rank(importance_array: np.ndarray) -> np.ndarray:
    """Return Borda scores (higher = more important)."""
    n = len(importance_array)
    order = np.argsort(importance_array)
    borda = np.zeros(n)
    borda[order] = np.arange(n)
    return borda

def compute_feature_importance(model_df, feature_list, target_col, models, fold_imps):
    """Cross-fold averaged importance + Borda count aggregation."""
    sub = model_df.dropna(subset=[target_col])
    avail_feats = [f for f in feature_list if f in sub.columns]
    X = sub[avail_feats].values
    y = sub[target_col].astype(int).values

    imp_records = {}

    lr_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     LogisticRegression(C=1.0, class_weight='balanced',
                                       solver='lbfgs', max_iter=2000,
                                       random_state=CONFIG['random_state'])),
    ])
    lr_pipe.fit(X, y)
    lr_coefs = np.abs(lr_pipe.named_steps['clf'].coef_[0])
    imp_records['LR_coef'] = lr_coefs

    for model_name in ['RandomForest', 'XGBoost']:
        folds = fold_imps.get(model_name, [])
        if len(folds) == 0:
            continue
        # Pad to feature_list length (some folds may use subset)
        mean_imp = np.mean(np.vstack(folds), axis=0)
        if len(mean_imp) == len(avail_feats):
            imp_records[f'{model_name}_imp'] = mean_imp

    if not imp_records:
        return pd.DataFrame()

    borda_scores = np.zeros(len(avail_feats))
    for imp_arr in imp_records.values():
        borda_scores += borda_rank(imp_arr)

    imp_df = pd.DataFrame({'feature': avail_feats, 'borda_score': borda_scores})
    for name, arr in imp_records.items():
        imp_df[name] = arr
    imp_df = imp_df.sort_values('borda_score', ascending=False).reset_index(drop=True)
    return imp_df

if best_target and best_target in all_fold_imps:
    tgt_col = f'next_{best_target}'
    print(f'Computing feature importance for: {CONFIG["targets"][best_target]["label"]}')
    imp_df = compute_feature_importance(
        model_df, feature_list, tgt_col, MODELS, all_fold_imps[best_target]
    )
    print('\nTop 20 features (Borda count):')
    print(imp_df.head(20)[['feature', 'borda_score']].to_string(index=False))
    imp_df.to_csv(CONFIG['output_dir'] / 'feature_importance.csv', index=False)
else:
    imp_df = pd.DataFrame()
    print('No feature importance available (no valid targets ran)')

In [ ]:
if HAS_SHAP and best_target and len(imp_df) > 0:
    from sklearn.impute import SimpleImputer as SI

    tgt_col = f'next_{best_target}'
    sub = model_df.dropna(subset=[tgt_col])
    avail_feats = [f for f in feature_list if f in sub.columns]
    X_full = sub[avail_feats].values
    X_full = SI(strategy='median').fit_transform(X_full)

    rf_model = RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_leaf=8,
        class_weight='balanced', random_state=CONFIG['random_state']
    )
    rf_model.fit(X_full, sub[tgt_col].astype(int).values)

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_full)

    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_full, feature_names=avail_feats, show=False, max_display=20)
    plt.title(f'SHAP — {CONFIG["targets"][best_target]["label"]}')
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {CONFIG['output_dir'] / 'shap_summary.png'}")
else:
    print('SHAP skipped (not installed or no valid results)')

## 9. Visualisations

Generates bar charts of model performance metrics across targets, a feature-type distribution pie chart, and a ranked feature importance chart.

In [ ]:
if len(combined) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    metrics = ['bal_acc', 'f1_macro', 'roc_auc']
    metric_labels = ['Balanced Accuracy', 'F1 Macro', 'ROC AUC']
    summary_reset = summary.reset_index()

    for ax, metric, mlabel in zip(axes, metrics, metric_labels):
        pivot = summary_reset.pivot(index='target', columns='model', values=metric)
        pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
        ax.set_title(mlabel, fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='Chance')
        ax.legend(fontsize=7, ncol=2)
        ax.tick_params(axis='x', rotation=20)
        for bar in ax.patches:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                        f'{h:.2f}', ha='center', va='bottom', fontsize=6)

    plt.suptitle('eBATTLE — Model Performance (mean across CV folds)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'target_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

if feature_list:
    prefixes = ['recent_', 'cumul_mean_', 'cumul_std_', 'trend_', 'delta_', 'meta']
    counts = {}
    for pfx in prefixes[:-1]:
        counts[pfx] = sum(1 for f in feature_list if f.startswith(pfx))
    counts['meta'] = len(feature_list) - sum(counts.values())

    fig, ax = plt.subplots(figsize=(6, 6))
    wedges, texts, autotexts = ax.pie(
        counts.values(), labels=counts.keys(),
        autopct='%1.0f%%', startangle=140,
        colors=sns.color_palette('Set2', len(counts))
    )
    ax.set_title(f'Feature type distribution (n={len(feature_list)})', fontweight='bold')
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'feature_type_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

if len(imp_df) > 0:
    top20 = imp_df.head(20)
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, col, title in zip(axes,
                               ['borda_score', 'LR_coef'],
                               ['Borda Score (all models)', 'Logistic Regression |coef|']):
        if col not in top20.columns:
            ax.set_visible(False)
            continue
        top20_sorted = top20.sort_values(col)
        ax.barh(top20_sorted['feature'], top20_sorted[col],
                color=sns.color_palette('viridis', len(top20_sorted)))
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel(col)
        ax.tick_params(axis='y', labelsize=8)

    plt.suptitle(f'Feature Importance — {CONFIG["targets"][best_target]["label"]}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Weight Validation

Correlates per-user activity engagement rate with weight change over the trial using Spearman rank correlation and a Mann–Whitney U test.

In [ ]:
def validate_with_weight(cumul_df, df_panel, best_target):
    """Correlate per-user activity engagement rate with weight change."""
    if 'weight_kg' not in df_panel.columns:
        print('weight_kg not in panel — skipping weight validation')
        return

    tgt_col = f'next_{best_target}'

    if tgt_col in model_df.columns:
        engage = (
            model_df.dropna(subset=[tgt_col])
            .groupby('user_id')[tgt_col]
            .mean()
            .rename('engagement_rate')
        )
    else:
        print(f'Target {tgt_col} not found — skipping')
        return

    wt = (
        df_panel.dropna(subset=['weight_kg'])
        .groupby('user_id')
        .apply(lambda g: pd.Series({
            'weight_start': g.sort_values('date')['weight_kg'].iloc[0],
            'weight_end':   g.sort_values('date')['weight_kg'].iloc[-1],
            'n_weighings':  len(g),
        }))
    )
    wt['weight_change_kg']  = wt['weight_end'] - wt['weight_start']
    wt['weight_change_pct'] = (wt['weight_change_kg'] / wt['weight_start']) * 100

    merged = wt.join(engage).dropna()
    n = len(merged)
    print(f'\nWeight validation: {n} users with both activity and weight data')

    if n < 5:
        print('  Too few users for reliable statistics')
        return

    r, p = stats.spearmanr(merged['engagement_rate'], merged['weight_change_kg'])
    print(f'  Spearman r = {r:.3f}, p = {p:.3f}')

    med = merged['engagement_rate'].median()
    high = merged[merged['engagement_rate'] >= med]['weight_change_kg']
    low  = merged[merged['engagement_rate'] <  med]['weight_change_kg']
    u_stat, p_mw = stats.mannwhitneyu(high, low, alternative='less')
    print(f'  High engagement weight change: {high.mean():.2f} kg (n={len(high)})')
    print(f'  Low  engagement weight change: {low.mean():.2f} kg (n={len(low)})')
    print(f'  Mann-Whitney U p = {p_mw:.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(merged['engagement_rate'], merged['weight_change_kg'],
                    c='steelblue', alpha=0.7, s=60)
    m, b = np.polyfit(merged['engagement_rate'], merged['weight_change_kg'], 1)
    xs = np.linspace(0, 1, 100)
    axes[0].plot(xs, m * xs + b, 'r--', linewidth=1.5, label=f'r={r:.2f}')
    axes[0].axhline(0, color='grey', linestyle=':', linewidth=0.8)
    axes[0].set_xlabel('Activity Engagement Rate')
    axes[0].set_ylabel('Weight Change (kg)')
    axes[0].set_title('Engagement vs Weight Change')
    axes[0].legend()

    axes[1].bar(['High Engagement', 'Low Engagement'],
                [high.mean(), low.mean()],
                color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='white')
    axes[1].axhline(0, color='grey', linestyle=':', linewidth=0.8)
    axes[1].set_ylabel('Mean Weight Change (kg)')
    axes[1].set_title(f'High vs Low Engagement (p={p_mw:.3f})')

    plt.suptitle('Weight Validation (exploratory — small N)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(CONFIG['output_dir'] / 'weight_validation.png', dpi=150, bbox_inches='tight')
    plt.show()

    merged.to_csv(CONFIG['output_dir'] / 'weight_validation_merged.csv')
    return merged

if best_target:
    wt_merged = validate_with_weight(cumul_df, df_panel, best_target)

## 11. Save Outputs

Saves the final feature dataset and daily panel as versioned Parquet files and prints a dataset summary.

In [ ]:
from datetime import date as dt_date

today = dt_date.today().strftime('%Y%m%d')

parquet_path = CONFIG['output_dir'] / f'feature_dataframe_v3_{today}.parquet'
model_df.to_parquet(parquet_path, index=False)
print(f'Saved parquet : {parquet_path}')

panel_path = CONFIG['output_dir'] / f'daily_panel_v3_{today}.parquet'
df_panel.to_parquet(panel_path, index=False)
print(f'Saved panel   : {panel_path}')

print('\n' + '='*60)
print('THESIS DATASET SUMMARY')
print('='*60)
print(f'Total users          : {model_df["user_id"].nunique()}')
print(f'  Pilot 1            : {model_df[model_df["pilot_id"]==1]["user_id"].nunique()}')
print(f'  Pilot 2            : {model_df[model_df["pilot_id"]==2]["user_id"].nunique()}')
print(f'Total user-weeks     : {len(model_df):,}')
print(f'Final feature count  : {len(feature_list)}')
print(f'Date range           : {df_panel["date"].min().date()} → {df_panel["date"].max().date()}')
print(f'Phase 1 weeks        : {CONFIG["phase1_week_range"]} (days {CONFIG["phase1_days"]})')
print(f'Phase 2 weeks        : {CONFIG["phase2_week_range"]} (days {CONFIG["phase2_days"]})')
print(f'Day 1 source         : {CONFIG["day1_source"]}')
print(f'Best target (ROC AUC): {best_target} = {best_auc:.3f}')
print('\nTargets class balance:')
for tgt_name, tgt_cfg in CONFIG['targets'].items():
    col = f'next_{tgt_name}'
    if col in model_df.columns:
        vc = model_df[col].value_counts(normalize=True)
        print(f'  {tgt_cfg["label"]:<35s}: active={vc.get(1.0,0):.1%}')
print('='*60)